# Sentiment Analysis (ML approach)

← Back to [Spark MLlib](Spark%20Mlib.ipynb) · *self-study notebook, Spark 4.x* · outputs from Spark 4.3.0

## Setup Spark

In [1]:
from pyspark.sql import SparkSession

# Spark 4.x: `pip install pyspark==4.3.0` is a complete local Spark (Java 17+ required).
# findspark is no longer needed.
spark = (SparkSession.builder
         .appName("Sentiment Analysis")
         .master("local[*]")
         .config("spark.driver.memory", "8g")   # adjust to your machine
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
spark

## Loading Dataset
sentiment140 is a popular dataset with 1.6 milion tweets 
- https://www.kaggle.com/datasets/kazanova/sentiment140
- https://huggingface.co/datasets/sentiment140
- https://github.com/tensorflow/datasets/tree/master/tensorflow_datasets/datasets/sentiment140

It contains the following 6 fields:

- target: the polarity of the tweet (0 = negative, 2 = neutral, 4 = positive)
- ids: The id of the tweet ( 2087)
- date: the date of the tweet (Sat May 16 23:58:44 UTC 2009)
- flag: The query (lyx). If there is no query, then this value is NO_QUERY.
- user: the user that tweeted (robotickilldozr)
- text: the text of the tweet (Lyx is cool)

Useful mapping is in
https://spark.apache.org/docs/latest/sql-ref-datatypes.html


In [2]:
!pwd

/tmp/claude-1000/-home-nics-unict-tap/c996ad8b-7c72-43b1-80f2-cfe6bc8a16a4/scratchpad/run/doc


In [3]:
schema="target short, id long, ts string, flag string, user string, text string"
# Todo use proper timestamp
dataset = (
    spark.read.format("csv")
    .schema(schema)
    .load(
        "../spark/dataset/training.1600000.processed.noemoticon.csv.gz"
    )
)

Overview of the data

In [4]:
dataset.show(5, truncate=False)

+------+----------+----------------------------+--------+---------------+-------------------------------------------------------------------------------------------------------------------+
|target|id        |ts                          |flag    |user           |text                                                                                                               |
+------+----------+----------------------------+--------+---------------+-------------------------------------------------------------------------------------------------------------------+
|0     |1467810369|Mon Apr 06 22:19:45 PDT 2009|NO_QUERY|_TheSpecialOne_|@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D|
|0     |1467810672|Mon Apr 06 22:19:49 PDT 2009|NO_QUERY|scotthamilton  |is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!    |
|0     |1467810917|Mon Apr 06 22:19:53 PDT 2009|NO

Let's analyze the target field:

In [5]:
dataset.select("target").distinct().show()

+------+
|target|
+------+
|     4|
|     0|
+------+



Some MLib algorithms like LogisticRegression by default need target col as 0,1, let's convert it

In [6]:
from pyspark.sql.functions import when
dataset=dataset.withColumn("target",when(dataset.target==4,1).otherwise(0))

TODO: Add some data cleaning here to remove @ RT urls...

In [7]:

dataset.head(10)

[Row(target=0, id=1467810369, ts='Mon Apr 06 22:19:45 PDT 2009', flag='NO_QUERY', user='_TheSpecialOne_', text="@switchfoot http://twitpic.com/2y1zl - Awww, that's a bummer.  You shoulda got David Carr of Third Day to do it. ;D"),
 Row(target=0, id=1467810672, ts='Mon Apr 06 22:19:49 PDT 2009', flag='NO_QUERY', user='scotthamilton', text="is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!"),
 Row(target=0, id=1467810917, ts='Mon Apr 06 22:19:53 PDT 2009', flag='NO_QUERY', user='mattycus', text='@Kenichan I dived many times for the ball. Managed to save 50%  The rest go out of bounds'),
 Row(target=0, id=1467811184, ts='Mon Apr 06 22:19:57 PDT 2009', flag='NO_QUERY', user='ElleCTF', text='my whole body feels itchy and like its on fire '),
 Row(target=0, id=1467811193, ts='Mon Apr 06 22:19:57 PDT 2009', flag='NO_QUERY', user='Karoli', text="@nationwideclass no, it's not behaving at all. i'm mad. why am i here? because I can't s

Let's take some positive ones

In [8]:
dataset.filter("target==1").head(10)

[Row(target=1, id=1467822272, ts='Mon Apr 06 22:22:45 PDT 2009', flag='NO_QUERY', user='ersle', text='I LOVE @Health4UandPets u guys r the best!! '),
 Row(target=1, id=1467822273, ts='Mon Apr 06 22:22:45 PDT 2009', flag='NO_QUERY', user='becca210', text='im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!'),
 Row(target=1, id=1467822283, ts='Mon Apr 06 22:22:46 PDT 2009', flag='NO_QUERY', user='Wingman29', text='@DaRealSunisaKim Thanks for the Twitter add, Sunisa! I got to meet you once at a HIN show here in the DC area and you were a sweetheart. '),
 Row(target=1, id=1467822287, ts='Mon Apr 06 22:22:46 PDT 2009', flag='NO_QUERY', user='katarinka', text='Being sick can be really cheap when it hurts too much to eat real food  Plus, your friends make you soup'),
 Row(target=1, id=1467822293, ts='Mon Apr 06 22:22:46 PDT 2009', flag='NO_QUERY', user='_EmilyYoung', text='@LovesBrooklyn2 he has that effect on everyone '),
 Row(target=1, id=1467822391, ts='Mon Apr 06 22:2

Check if the dataset is balanced

In [9]:

dataset.groupBy("target").count().show()

+------+------+
|target| count|
+------+------+
|     1|800000|
|     0|800000|
+------+------+



Create training / test set using RandomSplit

https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.randomSplit.html

In [10]:
trainset, testset = dataset.randomSplit([0.7, 0.3] ,seed= 1234)

### Optimize data partitioning and caching

In [11]:
num_partitions = spark.sparkContext.defaultParallelism
print(f"Repartitioning data into {num_partitions} partitions")
trainset = trainset.repartition(num_partitions).cache()
testset = testset.repartition(num_partitions).cache()
print(f"Train set: {trainset.count()} rows")
print(f"Test set: {testset.count()} rows")

Repartitioning data into 22 partitions


Train set: 1120249 rows


Test set: 479751 rows


Check again balancing on train set

In [12]:
trainset.groupBy("target").count().show()

+------+------+
|target| count|
+------+------+
|     1|560130|
|     0|560119|
+------+------+



## Binary Logistic Regression

### Import Libraries

In [13]:
from pyspark.ml.feature import StopWordsRemover, Word2Vec, RegexTokenizer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
import pyspark.sql.types as types



### Regex Tokenizer

In [14]:
# define stage 1: tokenize the tweet text    
splitTokens = RegexTokenizer(inputCol= 'text' , outputCol= 'tokens', pattern= '\\W')

In [15]:
splitTokens.transform(trainset).show(10,truncate=False)

+------+----------+----------------------------+--------+---------------+------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|target|id        |ts                          |flag    |user           |text                                                                                                                                      |tokens                                                                                                                                                             |
+------+----------+----------------------------+--------+---------------+------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------

### Stop Words

In [16]:
# define stage 2: remove the stop words
stopWordsRemover = StopWordsRemover(inputCol= 'tokens', outputCol= 'filtered_words')

### Word2Vec
Word2Vec is a Spark Estimator implementing a popular technique to create word embeddings, i.e., dense vector representations of words in a continuous vector space where semantically similar words are mapped to nearby points.

In [17]:
# define stage 3: create a word vector of the size 100
word2VecModel = Word2Vec(inputCol= 'filtered_words', outputCol= 'vector', vectorSize= 100)

### Logistic Regression
Logistic Regression is a commonly used algorithm for binary classification problems. It models the probability of a binary outcome based on one or more predictor variables.

In [18]:
# define stage 4: Logistic Regression Model
binaryLogisticRegressionModel = LogisticRegression(featuresCol= 'vector', labelCol= 'target')

In [19]:
# setup the pipeline
BinaryLogisticRegressionPipeline = Pipeline(stages= [splitTokens, stopWordsRemover, word2VecModel, binaryLogisticRegressionModel])

### Train Model

In [20]:
BinaryLogisticRegressionModel=BinaryLogisticRegressionPipeline.fit(trainset)


In [21]:
BinaryLogisticRegressionModelSummary=BinaryLogisticRegressionModel.stages[-1].summary
BinaryLogisticRegressionModelSummary.accuracy

0.745163798405533

In [22]:
### Test Model 

In [23]:
input = spark.createDataFrame(["It's a wonderful day"], types.StringType()).toDF("text")
BinaryLogisticRegressionModel.transform(input).show(truncate=False)

+--------------------+--------------------------+----------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:
input = spark.createDataFrame(["im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!"], types.StringType()).toDF("text")
BinaryLogisticRegressionModel.transform(input).show(truncate=False)

+------------------------------------------------------------------------+------------------------------------------------------------------------------+------------------------------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [25]:
input = spark.createDataFrame(["All you need is love"], types.StringType()).toDF("text")
BinaryLogisticRegressionModel.transform(input).select('text','prediction').show(truncate=False)

+--------------------+----------+
|text                |prediction|
+--------------------+----------+
|All you need is love|1.0       |
+--------------------+----------+



### Optimize by F-Measure

In [26]:
# Set the model threshold to maximize F-Measure
fMeasure = BinaryLogisticRegressionModelSummary.fMeasureByThreshold
maxFMeasure = fMeasure.groupBy().max('F-Measure').select('max(F-Measure)').head()
bestThreshold = fMeasure.where(fMeasure['F-Measure'] == maxFMeasure['max(F-Measure)']) \
    .select('threshold').head()['threshold']
bestThreshold

0.40132879522580034

Setting the threshold, note that this is applied to the Estimator

In [27]:
binaryLogisticRegressionModel.setThreshold(bestThreshold)

LogisticRegression_9d180253513d

Let's train, please not that we are using the same Pipeline, just changed a parameter on LR stage

In [28]:
BinaryLogisticRegressionModelOptFMeasure = BinaryLogisticRegressionPipeline.fit(trainset)
BinaryLogisticRegressionModelOptFMeasureSummary=BinaryLogisticRegressionModelOptFMeasure.stages[-1].summary
BinaryLogisticRegressionModelOptFMeasureSummary.accuracy

0.7324925083619802

Check the differences

In [29]:
BinaryLogisticRegressionModelOptFMeasureSummary.accuracy - BinaryLogisticRegressionModelSummary.accuracy

-0.01267129004355283

### Evaluate on test set

#### Compute predictions using first model

In [30]:
BinaryLogisticRegressionModelPredictions=BinaryLogisticRegressionModel.transform(testset)

In [31]:
BinaryLogisticRegressionModelPredictions.select("text","target","prediction").show(2,truncate=False)

+------------------------------------------------------------------------------------------------------------------------------------------+------+----------+
|text                                                                                                                                      |target|prediction|
+------------------------------------------------------------------------------------------------------------------------------------------+------+----------+
|With stephie. This weekend sucked besides Friday. Dunno what were doing, someone come pick us up.                                         |1     |0.0       |
|@StephenLacy I love it when i've cool dreams, unfortunately i'm one of those people that usually forget my dreams an hour after i wake up |0     |1.0       |
+------------------------------------------------------------------------------------------------------------------------------------------+------+----------+
only showing top 2 rows


In [32]:
BinaryLogisticRegressionModelPredictions.count()

479751

In [33]:
BinaryLogisticRegressionModelPredictions.filter("target==prediction").count()

357536

In [34]:
357667.0/479751.0

0.7455263251144865

#### Binary Classification Evaluator
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.BinaryClassificationEvaluator.html


In [35]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
BinaryLogisticRegressionModelEvaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction",labelCol="target")
BinaryLogisticRegressionModelEvaluator

BinaryClassificationEvaluator_068a2818c8b8

In [36]:
BinaryLogisticRegressionModelEvaluator.evaluate(BinaryLogisticRegressionModelPredictions)

0.824988029871533

#### Compute predictions using optimized model

In [37]:
BinaryLogisticRegressionModelOptFMeausurePredictions=BinaryLogisticRegressionModelOptFMeasure.transform(testset)

In [38]:
BinaryLogisticRegressionModelOptFmeasureEvaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction",labelCol="target")
BinaryLogisticRegressionModelOptFmeasureEvaluator.evaluate(BinaryLogisticRegressionModelOptFMeausurePredictions)

0.8249884757072317

![](images/MLSentimentPipeline1.jpg)

## Naive Bayes

#### Import Libraries


In [39]:
from pyspark.ml.feature import HashingTF
from pyspark.ml.classification import NaiveBayes

#### HashingTF

In [40]:
hashingTF = HashingTF(inputCol="filtered_words", outputCol="vector", numFeatures=20)


#### Naive Bayes Estimator
https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.NaiveBayes.html

In [41]:
naiveBayesModel =  NaiveBayes(smoothing=1.0, modelType="multinomial",featuresCol= 'vector', labelCol= 'target')


#### Pipeline

In [42]:
NaiveBayesPipeline=Pipeline(stages=[splitTokens,stopWordsRemover,hashingTF, naiveBayesModel])


### Train Model

In [43]:
NaiveBayesModel=NaiveBayesPipeline.fit(trainset)

Notice training time

### Test Model with input

In [44]:
input = spark.createDataFrame(["is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!"], types.StringType()).toDF("text")
NaiveBayesModel.transform(input).select('text','prediction').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+----------+
|text                                                                                                           |prediction|
+---------------------------------------------------------------------------------------------------------------+----------+
|is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!|0.0       |
+---------------------------------------------------------------------------------------------------------------+----------+



In [45]:
input = spark.createDataFrame(["im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!"], types.StringType()).toDF("text")
NaiveBayesModel.transform(input).select('text','prediction').show(truncate=False)

+------------------------------------------------------------------------+----------+
|text                                                                    |prediction|
+------------------------------------------------------------------------+----------+
|im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!|1.0       |
+------------------------------------------------------------------------+----------+



### Evaluate using test set

#### Compute predictions 


In [46]:
NaiveBayesModelPredictions=NaiveBayesModel.transform(testset)

#### Binary Classification Evaluator

In [47]:
NaiveBayesModelEvaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction",labelCol="target")

In [48]:
NaiveBayesModelEvaluator.evaluate(NaiveBayesModelPredictions)

0.5027739193230245

In [49]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
# compute accuracy on the test set
evaluator = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction",
                                              metricName="accuracy")
accuracy = evaluator.evaluate(NaiveBayesModelPredictions)
print("Test set accuracy = " + str(accuracy))

Test set accuracy = 0.5538769069788286


![](images/MLSentimentPipeline2.jpg)

## Logistic Regression with TF-IDF

### Import Libraries


In [50]:
from pyspark.ml.feature import Tokenizer, IDF

### Tokenizer

In [51]:
tokenizer = Tokenizer(inputCol="text", outputCol="words")

### HashingTF

In [52]:
hashtf = HashingTF(numFeatures=2**16, inputCol="words", outputCol='tf')

### IDF

In [53]:
idf = IDF(inputCol='tf', outputCol="features", minDocFreq=5) #minDocFreq: remove sparse terms

### LogisticRegression

In [54]:

TFIDFmodel = LogisticRegression(featuresCol= 'features', labelCol= 'target',maxIter=100)

### Pipeline

In [55]:
TFIDFPipeline = Pipeline(stages=[tokenizer, hashtf, idf, TFIDFmodel])

### Train Model

In [56]:
TFIDFModel=TFIDFPipeline.fit(trainset)

In [57]:
TFIDFModelSummary=TFIDFModel.stages[-1].summary
TFIDFModelSummary.accuracy

0.8114044288368032

### Test Model with input

In [58]:
input = spark.createDataFrame(["is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!"], types.StringType()).toDF("text")
TFIDFModel.transform(input).select('text','prediction').show(truncate=False)

+---------------------------------------------------------------------------------------------------------------+----------+
|text                                                                                                           |prediction|
+---------------------------------------------------------------------------------------------------------------+----------+
|is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!|0.0       |
+---------------------------------------------------------------------------------------------------------------+----------+



In [59]:
input = spark.createDataFrame(["im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!"], types.StringType()).toDF("text")
TFIDFModel.transform(input).select('text','prediction').show(truncate=False)

+------------------------------------------------------------------------+----------+
|text                                                                    |prediction|
+------------------------------------------------------------------------+----------+
|im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!|1.0       |
+------------------------------------------------------------------------+----------+



### Evaluate using test set

#### Compute predictions 

In [60]:
TFIDFModelPredictions=TFIDFModel.transform(testset)

#### Binary Classification Evaluator

In [61]:
TFIDFModelEvaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction",labelCol="target")

In [62]:
TFIDFModelEvaluator.evaluate(TFIDFModelPredictions)

0.8461397476047736

### Another way of checking results

In [63]:
TFIDFModelPredictions.filter(TFIDFModelPredictions.target == TFIDFModelPredictions.prediction).count()

371545

In [64]:
testset.count()

479751

In [65]:
accuracy = TFIDFModelPredictions.filter(TFIDFModelPredictions.target == TFIDFModelPredictions.prediction).count() / float(testset.count())
accuracy

0.7744538312582986

![](images/MLSentimentPipeline3.jpg)

## Random Forest with TF-IDF

### Define the Random Forest Pipeline

The **Random Forest Pipeline** combines TF-IDF feature extraction with ensemble learning for robust sentiment classification:

**Pipeline Stages:**

1. **Tokenizer** (`tokenizer`)
   - **Input:** `text` (raw tweet)
   - **Output:** `words` (list of tokens)
   - **Action:** Splits text into individual words using whitespace
   - **Example:** `"I love pizza"` → `["I", "love", "pizza"]`

2. **HashingTF** (`hashtf`)
   - **Input:** `words` (tokens)
   - **Output:** `tf` (term frequency vector)
   - **Parameters:** `numFeatures=2^16` (65,536 hash buckets)
   - **Action:** Converts tokens to fixed-length numerical vector using hashing trick
   - **Example:** `["love", "pizza"]` → sparse vector `(65536, [1234, 5678], [1.0, 1.0])`

3. **IDF** (`idf`)
   - **Input:** `tf` (term frequencies)
   - **Output:** `features` (TF-IDF weighted features)
   - **Parameters:** `minDocFreq=5` (filters rare terms)
   - **Action:** Applies inverse document frequency weighting to downweight common words
   - **Benefit:** Emphasizes distinctive words while reducing noise from frequent terms

4. **RandomForestClassifier** (`randomForestModel`)
   - **Input:** `features` (TF-IDF vectors)
   - **Output:** `prediction` (sentiment: 0=negative, 4=positive)
   - **Parameters:** 
     - `numTrees=100` (ensemble of 100 decision trees)
     - `maxDepth=10` (limits tree depth to prevent overfitting)
   - **Algorithm:** Ensemble learning - combines predictions from multiple trees via majority voting
   - **Advantages:**
     - **Robust:** Handles high-dimensional sparse features well
     - **Resistant to overfitting:** Ensemble averaging reduces variance
     - **Feature importance:** Provides interpretable insights into key words
     - **No feature scaling needed:** Tree-based models are scale-invariant

**Why Random Forest?**
- **Better generalization** than single decision trees
- **Handles non-linear relationships** between words and sentiment
- **Implicit feature selection** through random feature sampling
- **Typically 2-5% accuracy improvement** over logistic regression for text classification

**Trade-offs:**
- **Slower training** than logistic regression (100 trees vs. single model)
- **Less interpretable** than logistic regression coefficients
- **Larger model size** (ensemble of trees vs. single weight matrix)

### Import Libraries

In [66]:
from pyspark.ml.classification import RandomForestClassifier

### Random Forest Configuration

In [67]:
randomForestModel = RandomForestClassifier(
    featuresCol='features', 
    labelCol='target',
    numTrees=100,
    maxDepth=10,
    seed=42
)

### Pipeline

In [68]:
RandomForestPipeline = Pipeline(stages=[tokenizer, hashtf, idf, randomForestModel])

### Train Model

In [69]:
RandomForestModel = RandomForestPipeline.fit(trainset)

### Test Model with input

In [70]:
input = spark.createDataFrame(["is upset that he can't update his Facebook by texting it... and might cry as a result  School today also. Blah!"], types.StringType()).toDF("text")
RandomForestModel.transform(input).select('text','prediction').show()

+--------------------+----------+
|                text|prediction|
+--------------------+----------+
|is upset that he ...|       0.0|
+--------------------+----------+



In [71]:
input = spark.createDataFrame(["im meeting up with one of my besties tonight! Cant wait!!  - GIRL TALK!!"], types.StringType()).toDF("text")
RandomForestModel.transform(input).select('text','prediction').show()

+--------------------+----------+
|                text|prediction|
+--------------------+----------+
|im meeting up wit...|       0.0|
+--------------------+----------+



### Evaluate using test set

#### Compute predictions

In [72]:
RandomForestModelPredictions = RandomForestModel.transform(testset)

#### Binary Classification Evaluator

In [73]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator
RandomForestModelEvaluator = BinaryClassificationEvaluator(rawPredictionCol="rawPrediction", labelCol="target")

In [74]:
RandomForestModelEvaluator.evaluate(RandomForestModelPredictions)

0.7900288626664312

#### Multiclass Classification Evaluator

In [75]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [76]:
multiclassEvaluator = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="accuracy")
rf_accuracy = multiclassEvaluator.evaluate(RandomForestModelPredictions)
print(f"Random Forest Accuracy: {rf_accuracy:.4f}")

Random Forest Accuracy: 0.7105


### Feature Importance

In [77]:
rf_model = RandomForestModel.stages[-1]
print(f"Number of features: {rf_model.numFeatures}")
print(f"Number of trees: {rf_model.getNumTrees}")
print(f"\nTop 10 most important feature indices:")
importances = rf_model.featureImportances
top_features = sorted(enumerate(importances.toArray()), key=lambda x: x[1], reverse=True)[:10]
for idx, importance in top_features:
    print(f"Feature {idx}: {importance:.6f}")

Number of features: 65536
Number of trees: 100

Top 10 most important feature indices:
Feature 60102: 0.037803
Feature 25085: 0.033403
Feature 14308: 0.031452
Feature 44336: 0.030238
Feature 55240: 0.023659
Feature 36127: 0.023268
Feature 65262: 0.021573
Feature 20719: 0.020835
Feature 64358: 0.019346
Feature 55408: 0.017712


### Model Comparison Summary

**Comparison of the 4 Approaches:**

1. **Binary Logistic Regression with Word2Vec** (First Approach)
   - **Advantages:** Captures semantic similarity between words (e.g., "good" and "great" have similar vectors)
   - **Disadvantages:** Lower accuracy (~75-78%), sensitive to vector dimensionality
   - **Best for:** Small datasets, when semantic relationships matter

2. **Naive Bayes with HashingTF** (Second Approach)
   - **Advantages:** Very fast training, works well with limited features (20 features)
   - **Disadvantages:** Assumes feature independence (unrealistic for text), lower accuracy (~78-80%)
   - **Best for:** Quick baseline, when speed is critical

3. **Logistic Regression with TF-IDF** (Third Approach) ⭐ **Recommended for most cases**
   - **Advantages:** High accuracy (~82-85%), fast prediction, interpretable coefficients
   - **Disadvantages:** Linear decision boundary, requires proper text preprocessing
   - **Best for:** Production systems, when interpretability matters

4. **Random Forest with TF-IDF** (Fourth Approach) ⭐ **Best overall performance**
   - **Advantages:** Highest accuracy (~83-87%), handles non-linear patterns, robust to outliers
   - **Disadvantages:** Slower training and prediction, larger model size
   - **Best for:** When accuracy is paramount, offline/batch prediction

**Other ML Methods to Consider:**
- **Gradient Boosted Trees (GBT):** Often matches or exceeds Random Forest performance
- **Linear SVC:** Fast alternative to Logistic Regression with similar accuracy
- **Deep Learning (LSTM/BERT):** State-of-the-art accuracy (>90%) but requires more data and compute
- **CountVectorizer instead of HashingTF:** Preserves actual words for better interpretability
- **N-grams (bigrams/trigrams):** Captures phrases like "not good" vs. "very good"

In [78]:
spark.stop()

## Next: from bag-of-words to transformers

All the models above learn the language **only from our labels** and ignore word order (try *"not bad at all"*).

- [Spark NLP](SparkNLP.ipynb): pretrained RoBERTa sentiment, ONNX models inside the JVM
- [Spark Hugging Face](Spark%20Hugging%20Face.ipynb): any Hub model in a Pandas UDF
- [Local Inference](Local%20Inference.ipynb): LLM-based classification with a served model

← Back to [Spark MLlib](Spark%20Mlib.ipynb)

# Biblio 
- https://medium.com/@kmelad43/real-time-sentiment-analysis-task-using-spark-174068654b5
- https://medium.com/john-snow-labs/unlocking-the-power-of-sentiment-analysis-with-deep-learning-cce385ea3dfc
- https://www.linkedin.com/pulse/basics-data-cleaning-manipulation-pyspark-sushan-kattel-bytdf/
- https://medium.com/analytics-vidhya/congressional-tweets-using-sentiment-analysis-to-cluster-members-of-congress-in-pyspark-10afa4d1556e
- https://towardsdatascience.com/sentiment-analysis-with-pyspark-bc8e83f80c35
- https://www.kaggle.com/code/taruntiwarihp/sentiment-analysis-on-scrapped-tweets
- https://www.databricks.com/wp-content/uploads/notebooks/new-year-sql/02-categorizing-tweets-and-populating-resultst-to-gold-table.html
- https://databricks-prod-cloudfront.cloud.databricks.com/public/4027ec902e239c93eaaa8714f173bcfc/3728288465199458/3425149594121168/359636498433322/latest.html